In [2]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-5-5"

In [3]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    for block in message.content:
        if block.type == "text":
            return block.text

## 課題: 構造化出力で、EventBridge ルールの JSON を生成する

- 対応: L08「構造化データ」。ページの方法(プリフィルとストップシーケンス)は、現行モデルでは使えない可能性が高いため、公式の構造化出力(`output_config.format`)を使う。
- 今回、新しく覚えること: `create()` に、JSON スキーマを渡して、応答の形を制限する。
- 目標: 説明文なしの、生の JSON(`source` と `detail-type`)を取り出して、`json.loads()` で、Python の辞書にする。
- 注意: `chat()` は、`output_config` を受け取らないので、今回は、`client.messages.create()` を直接呼ぶ。
- API を呼ぶので、少額の料金がかかる。モデルは、`claude-haiku-5-5`、`max_tokens` は、小さめにする。
- ヒントは、下の「ヒント」のセルにある。まず、見ずに、試してみる。

In [4]:
messages = []
add_user_message(messages, "EC2 インスタンスの状態が running になったときに反応する、短い EventBridge ルールを JSON で作ってください")

# TODO 1: JSON スキーマを書く
#   - 全体は object。properties に "source" と "detail-type" の2つ。
#   - どちらも、文字列の配列。
schema = {
    "type": "object",  # TODO: 全体の type
    "properties": {
        "source": {
            "type": "array",  # TODO: 配列を表す type
            "items": {"type": "string"},  # TODO: 配列の中身の type
        },
        "detail-type": {
            "type": "array",  # TODO
            "items": {"type": "string"},  # TODO
        },
    },
    "required": ["source", "detail-type"],  # TODO: 必須にするキー
    "additionalProperties": False,  # TODO: True か False か
}

In [8]:
# TODO 2: create() に、output_config を渡して呼び出す
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    output_config={"format": {"type": "json_schema", "schema": schema}},  # TODO: {"format": {...}} の形で、schema を渡す
)

# 停止理由も、あわせて確認する
print(response.stop_reason)

end_turn


In [ ]:
import json

# TODO 3: response.content から、type が "text" のブロックを探して、本文を text に入れる
def get_text_from_response(response):
    for block in response.content:
        if block.type == "text":
            return block.text
    return None

text = get_text_from_response(response)

# TODO 4: text を、json.loads() で、辞書にして、表示する
rule = json.loads(text) 
print(rule)

{'source': ['aws.ec2'], 'detail-type': ['EC2 Instance State-change Notification']}


## ヒント(困ったときだけ開く)

**TODO 1(スキーマ)**
- JSON スキーマの型の名前は、`"object"`、`"array"`、`"string"` など。
- 配列の中身の型は、`"items"` に書く。
- 「必須」は、`"required"` に、キーの名前を、リストで並べる(`"detail-type"` のように、ハイフン付きのキーも、そのまま文字列で書く)。
- `additionalProperties` は、公式ドキュメントで、オブジェクトには、どちらにする必要があると書かれているか。[Structured outputs](https://platform.claude.com/docs/en/build-with-claude/structured-outputs) の「Key limitations」を見る。

**TODO 2(`output_config`)**
- 形は、`{"format": {"type": "json_schema", "schema": ...}}` の3段。
- `schema` には、TODO 1 で作った変数 `schema` を渡す。
- `model` や `messages` の渡し方は、これまでの `create()` と同じ。

**TODO 3(取り出し)**
- L03 の質問2で答えた、`for` と `if` の形を使う。
- `content[0]` を決め打ちにしない。このモデルは、`ThinkingBlock` を先に付けることがある(L07 の出力を思い出す)。
- ブロックの種類は、`block.type` で見る。

**TODO 4(JSON にする)**
- `json.loads(text)` で、辞書になる。
- ページでは、改行を取るために `text.strip()` を使っていた。今回は、構造化出力なので、必要ないことが多い。必要かどうかは、出力を見て決める。

**エラーが出たら**
- 最後の行を読む。400 の場合、`message` の文面に、どの項目が悪いかが書かれていることが多い(`system` のときの、`system: Input should be a valid array` のように)。
- スキーマに、使えない機能を書くと、400 になる(数値の最小・最大、文字列の長さの制約など)。

## 振り返り(自分の言葉で書く)

1. ページの方法(プリフィルとストップシーケンス)は、なぜ、現行モデルでは使えない可能性が高いか。
2. 構造化出力は、ページの方法と比べて、何が保証されるか。何を、まだ自分で確認する必要があるか(`stop_reason` を見る理由)。
3. `content[0]` ではなく、`type` を見て取り出す理由。